# Calculating PV adapted from Kate's code 

"PV is calculated from f/H, where f is the Coriolis parameter and H is layer thickness,
assuming relative vorticity is negligable compared to planetary vorticity and velocity gradients." - Kate

In [3]:
import pandas as pd
import numpy as np
import gsw

In [2]:
# Get data into dataframe
df = pd.read_csv('../WMA_fractions_v2.csv', skiprows=1)

In [5]:
df['sigma0'] = gsw.sigma0(df['Absolute_Salinity_[PSU]'],df['Conservative_Temperature_[deg_C]'])

In [10]:
def normalize(arr, range_min, range_max):
    norm_arr = []
    diff = range_max - range_min
    diff_arr = max(arr) - min(arr)
    for i in arr:
        if diff_arr == 0:
            norm_arr.append(np.nan)  # Handle division by zero
        else:
            temp = (((i - min(arr)) * diff) / diff_arr) + range_min
            norm_arr.append(temp)
    return np.array(norm_arr)

def calculate_h_and_PV_from_rho(profile):

    # Remove mixed layer
    # Check the profile starts shallower than 10m
    if profile['Depth_[m]'].iloc[0]< 10:
        # Find the surface density
        surface_density = profile['sigma0'].iloc[0]
        # Find the maximum depth at which potential density is within 0.1kg/m3 of surface density
        mld = profile.loc[(profile['sigma0'] <= surface_density + 0.1), 'Depth_[m]'].max()
        # Remove depths in profile shallower than the mixed layer depth
        profile = profile[profile['Depth_[m]'] >= mld]

    # Check if the difference in sigma0 is zero
    if np.diff(profile['sigma0']).any() == 0:
        return None  # Return None to indicate skipping this nprof

    # Normalize profile
    rho_norm = normalize(profile['sigma0'], 10, 100)
    
    # Calculate density layer thickness
    rho_0 = np.nanmin(rho_norm)
    rholog = np.log(rho_norm / rho_0)
    profile['h_d'] = profile['Depth_[m]'] / rholog
    
    # Calculate Coriolis parameter
    profile['f'] = gsw.f(profile['Latitude_[deg_N]'])
    
    # Calculate Potential Vorticity
    profile['PV'] = profile['f'] / profile['h_d']

    return profile

In [11]:
df.head()

,Arctic_Surface_Water_[fraction],Modified_summer_Pacific_Water_[fraction],Summer_Pacific_Water_[fraction],Winter_Pacific_Water_[fraction],Norwegian_Current_Water_[fraction],Atlantic_Water_[fraction],Brine-enriched_Water_[fraction],Conservative_Temperature_[deg_C],Absolute_Salinity_[PSU],Latitude_[deg_N],Depth_[m],Longitude_[deg_E],Dissolved_Oxygen_[micro_mol_per_kg],Datetime_[UTC],Source,Profile_Number,sigma0
0,0.000000,0.000000,0.002680,0.000000,0.997320,0.000000,0.000000,9.222,35.352,63.315,15.0,-0.075,NaN,2009-08-09,argo,1,27.227478
1,0.000154,0.000148,0.000496,0.000102,0.995409,0.003583,0.000107,7.958,35.402,63.317,105.0,-0.017,NaN,2009-08-09,argo,1,27.464854
2,0.000322,0.000295,0.000327,0.000134,0.979148,0.019728,0.000046,7.831,35.402,63.318,115.0,-0.090,NaN,2009-08-09,argo,1,27.483868
3,0.000274,0.000256,0.000259,0.000111,0.965319,0.033748,0.000033,7.724,35.400,63.309,125.0,-0.068,NaN,2009-08-09,argo,1,27.498193
4,0.000124,0.000116,0.000117,0.000051,0.944119,0.055455,0.000017,7.550,35.395,63.368,135.0,-0.018,NaN,2009-08-09,argo,1,27.519857


In [12]:
# Apply the function to each 'nprof' of each 'source'
df = df.groupby(['Source','Profile_Number']).apply(calculate_h_and_PV_from_rho)
df = df.drop(['Source','Profile_Number'], axis=1)
df.reset_index(inplace=True)
df.head()

/var/folders/v0/0lp_zzrd33sgh7y2jyh9zzdm0000gn/T/ipykernel_26910/2828882217.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(['Source','Profile_Number']).apply(calculate_h_and_PV_from_rho)


,Source,Profile_Number,level_2,Arctic_Surface_Water_[fraction],Modified_summer_Pacific_Water_[fraction],Summer_Pacific_Water_[fraction],Winter_Pacific_Water_[fraction],Norwegian_Current_Water_[fraction],Atlantic_Water_[fraction],Brine-enriched_Water_[fraction],...,Absolute_Salinity_[PSU],Latitude_[deg_N],Depth_[m],Longitude_[deg_E],Dissolved_Oxygen_[micro_mol_per_kg],Datetime_[UTC],sigma0,h_d,f,PV
0,MOSAiC,1,11402581,0.008253,0.007459,0.008313,0.006659,0.105124,0.833460,0.030732,...,34.924,85.384,145.0,129.291,304.16,2019-10-24,27.863908,63.857751,0.000145,0.000002
1,MOSAiC,1,11402582,0.252448,0.000034,0.000123,0.003134,0.000095,0.000029,0.744138,...,32.562,85.384,25.0,129.291,386.28,2019-10-24,26.078078,inf,0.000145,0.000000
2,MOSAiC,1,11402583,0.043265,0.000424,0.001514,0.004868,0.001349,0.000514,0.948066,...,34.116,85.384,35.0,129.291,365.96,2019-10-24,27.333592,17.847866,0.000145,0.000008
3,MOSAiC,1,11402584,0.003355,0.003118,0.003227,0.002834,0.167735,0.802484,0.017247,...,35.045,85.384,305.0,129.291,299.51,2019-10-24,27.928485,132.459817,0.000145,0.000001
4,MOSAiC,1,11402585,0.026967,0.000074,0.000264,0.000347,0.000205,0.000063,0.972079,...,34.248,85.384,45.0,129.291,363.85,2019-10-24,27.441118,22.145357,0.000145,0.000007


In [14]:
df.drop(columns=['sigma0','h_d','f'], inplace=True)

In [15]:
df.head()

,Source,Profile_Number,level_2,Arctic_Surface_Water_[fraction],Modified_summer_Pacific_Water_[fraction],Summer_Pacific_Water_[fraction],Winter_Pacific_Water_[fraction],Norwegian_Current_Water_[fraction],Atlantic_Water_[fraction],Brine-enriched_Water_[fraction],Conservative_Temperature_[deg_C],Absolute_Salinity_[PSU],Latitude_[deg_N],Depth_[m],Longitude_[deg_E],Dissolved_Oxygen_[micro_mol_per_kg],Datetime_[UTC],PV
0,MOSAiC,1,11402581,0.008253,0.007459,0.008313,0.006659,0.105124,0.833460,0.030732,0.837,34.924,85.384,145.0,129.291,304.16,2019-10-24,0.000002
1,MOSAiC,1,11402582,0.252448,0.000034,0.000123,0.003134,0.000095,0.000029,0.744138,-1.771,32.562,85.384,25.0,129.291,386.28,2019-10-24,0.000000
2,MOSAiC,1,11402583,0.043265,0.000424,0.001514,0.004868,0.001349,0.000514,0.948066,-1.752,34.116,85.384,35.0,129.291,365.96,2019-10-24,0.000008
3,MOSAiC,1,11402584,0.003355,0.003118,0.003227,0.002834,0.167735,0.802484,0.017247,1.309,35.045,85.384,305.0,129.291,299.51,2019-10-24,0.000001
4,MOSAiC,1,11402585,0.026967,0.000074,0.000264,0.000347,0.000205,0.000063,0.972079,-1.783,34.248,85.384,45.0,129.291,363.85,2019-10-24,0.000007


In [16]:
# Compute gradients of PV

# Group the DataFrame by profile
grouped = df.groupby(['Source', 'Profile_Number'])

# Define a function to compute the vertical gradient of PV within each group
def compute_gradient(group):
    delta_depth = group['Depth_[m]'].diff()  # Compute difference in depth between adjacent rows
    delta_PV = group['PV'].diff()  # Compute difference in PV between adjacent rows
    gradient_PV = delta_PV / delta_depth  # Compute gradient as change in PV divided by change in depth

    # Handle boundary conditions within each group
    gradient_PV.iloc[0] = (group['PV'].iloc[1] - group['PV'].iloc[0]) / (group['Depth_[m]'].iloc[1] - group['Depth_[m]'].iloc[0])
    gradient_PV.iloc[-1] = (group['PV'].iloc[-1] - group['PV'].iloc[-2]) / (group['Depth_[m]'].iloc[-1] - group['Depth_[m]'].iloc[-2])

    return gradient_PV

# Apply the function to each group and concatenate the results
gradients = grouped.apply(compute_gradient)

gradients = gradients.reset_index()
gradients = gradients.rename(columns={0: 'gradient_PV'})

df['dPVdz'] = gradients['gradient_PV']
df['dPVdz']

# Replace infinite values with NaN 
df['dPVdz'].replace([np.inf, -np.inf], np.nan, inplace=True)

/var/folders/v0/0lp_zzrd33sgh7y2jyh9zzdm0000gn/T/ipykernel_26910/719663736.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gradients = grouped.apply(compute_gradient)
/var/folders/v0/0lp_zzrd33sgh7y2jyh9zzdm0000gn/T/ipykernel_26910/719663736.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to 

In [17]:
df.to_csv('../WMA_fractions_v2_with_PV.csv', index=False)